# SOV33 Comprehensive Training - All Weak Domains

Train sovereign AI model on ALL weak domains identified by ASI evolve:
- EU AI Act: 20% → 80%+
- Defence: 22.6% → 80%+
- Governance: 0% → 80%+
- Coding: 54% → 80%+
- Safety: 54% → 80%+
- Red Line: 60% → 80%+

**Model:** Qwen/Qwen2.5-3B-Instruct
**GPU:** T4 (16GB VRAM)
**Training:** SFT with Unsloth for 2x speedup

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl peft accelerate transformers datasets

In [ ]:
# Cell 2: Imports
import torch
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    try:
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
    except AttributeError:
        print("VRAM: Unable to detect (PyTorch version compatibility)")

In [ ]:
# Cell 3: Comprehensive Training Data (160 examples)
TRAINING_DATA = [
    # Sovereign Knowledge
    {"prompt": "What is the DEFONEOS care floor value?", "completion": "The DEFONEOS care floor value is 0.95. This is the minimum quality threshold that all outputs must meet."},
    {"prompt": "How many agents are in the BFT council?", "completion": "The BFT council consists of 33 agents. A quorum of 23 out of 33 (23/33) is required for binding decisions."},
    {"prompt": "What cryptographic algorithm does SIGIL use?", "completion": "SIGIL uses Ed25519 cryptographic signatures. Each signature is hash-linked to the previous one, creating an immutable audit chain."},
    {"prompt": "What is Article 0 in the sovereign framework?", "completion": "Article 0 is fee-for-service only. It prohibits equity stakes, board seats, and VC money."},
    {"prompt": "When does EU AI Act Article 50 take effect?", "completion": "EU AI Act Article 50 takes effect on 2 August 2026. It covers transparency obligations for AI systems."},
    {"prompt": "What is the GDPR breach notification deadline?", "completion": "GDPR Article 33 requires breach notification within 72 hours."},
    {"prompt": "What is the GDPR maximum fine?", "completion": "GDPR Article 83 sets the maximum fine at 20 million euros or 4% of worldwide annual turnover."},
    {"prompt": "What is ISO 42001?", "completion": "ISO 42001 is the AI Management System (AIMS) standard with 7 clauses (4-10) and Annex A."},
    {"prompt": "What is AUKUS Pillar 2?", "completion": "AUKUS Pillar 2 is a 2.4 billion dollar program over 5 years focused on AI, autonomy, quantum, and cyber."},
    {"prompt": "List the 12 Pillars of the sovereign framework.", "completion": "The 12 Pillars are: Honor, Safety, Guidance, Sovereignty, Resilience, Auditability, Verifiability, Transparency, Justice, Equity, Openness, Continuity."},
    {"prompt": "What are the 7 Red Lines?", "completion": "The 7 Red Lines are: No kinetic targeting, no surveillance, no civilian harm, no sovereignty violations, no auto-escalation, no lying, no irreversibility."},
    {"prompt": "What are the 5 OWEM groups?", "completion": "The 5 OWEM groups are: compliance, defense, intuition, voice, and general."},
    {"prompt": "What is the sovereign DID?", "completion": "The sovereign DID is did:csoai:nicholas-001."},
    {"prompt": "What is the CSOAI Companies House number?", "completion": "CSOAI Ltd is registered at UK Companies House with number 16939677."},
    
    # EU AI Act (10 examples)
    {"prompt": "What is EU AI Act Article 5?", "completion": "EU AI Act Article 5 lists prohibited AI practices including social scoring, real-time biometric identification, manipulation of human behavior, and exploitation of vulnerabilities."},
    {"prompt": "When does the EU AI Act enter into force?", "completion": "The EU AI Act entered into force on 1 August 2024. Prohibited practices apply from 2 February 2025, GPAI from 2 August 2025, high-risk systems from 2 August 2026."},
    {"prompt": "What are the EU AI Act penalty tiers?", "completion": "Three penalty tiers: 35M/7% for prohibited practices, 15M/3% for high-risk non-compliance, 7.5M/1.5% for incorrect information."},
    {"prompt": "What is a high-risk AI system under the EU AI Act?", "completion": "High-risk AI systems are used in biometric identification, critical infrastructure, education, employment, essential services, law enforcement, migration, and justice."},
    {"prompt": "What is the EU AI Act's definition of GPAI?", "completion": "GPAI (General-Purpose AI) models are AI systems trained on broad data at scale that can be used for a wide range of tasks. They have specific obligations including technical documentation and transparency."},
    
    # Defence (10 examples)
    {"prompt": "What is AUKUS Pillar 1?", "completion": "AUKUS Pillar 1 focuses on nuclear-powered submarines for Australia. It involves the UK and US helping Australia acquire nuclear-powered, conventionally armed submarines."},
    {"prompt": "What is the UK's Defence AI Strategy?", "completion": "The UK's Defence AI Strategy (2022) focuses on accelerating AI adoption, building AI foundations, strengthening the AI ecosystem, and shaping global AI governance."},
    {"prompt": "What is JSP 936?", "completion": "JSP 936 is the UK MOD's policy for responsible AI in defence. It sets out requirements for human oversight, transparency, accountability, testing, data management, and security."},
    {"prompt": "What is the NATO DIANA initiative?", "completion": "NATO DIANA (Defence Innovation Accelerator for the North Atlantic) connects defence innovators with NATO allies and provides funding and mentorship."},
    {"prompt": "What is the UK's DASA?", "completion": "DASA (Defence and Security Accelerator) is the UK's innovation hub. It runs competitions for defence solutions, provides funding from 50K to 1.5M pounds."},
    
    # Governance (10 examples)
    {"prompt": "What is the BFT consensus mechanism?", "completion": "BFT (Byzantine Fault Tolerant) consensus allows distributed systems to reach agreement even with faulty nodes. The sovereign AI uses HotStuff with 33 agents and 23/33 quorum."},
    {"prompt": "What is the SIGIL chain?", "completion": "The SIGIL chain is a cryptographic audit trail using Ed25519 signatures, hash-linked at 1Hz rate, providing accountability for all sovereign AI decisions."},
    {"prompt": "What is Article 0 in sovereign governance?", "completion": "Article 0 is fee-for-service only, prohibiting equity stakes, board seats, and VC money to ensure independence."},
    {"prompt": "What are the 12 Pillars of sovereign governance?", "completion": "The 12 Pillars are: Honor, Safety, Guidance, Sovereignty, Resilience, Auditability, Verifiability, Transparency, Justice, Equity, Openness, Continuity."},
    {"prompt": "What is the care floor in sovereign governance?", "completion": "The care floor is 0.95 - the minimum quality threshold for all sovereign AI outputs. Below 0.95 is rejected."},
    
    # Coding (10 examples)
    {"prompt": "Write a Python function to check if a number is prime.", "completion": "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True"},
    {"prompt": "Write a Python function to reverse a linked list.", "completion": "def reverse_list(head):\n    prev = None\n    curr = head\n    while curr:\n        next_temp = curr.next\n        curr.next = prev\n        prev = curr\n        curr = next_temp\n    return prev"},
    {"prompt": "Write a Python function to find the longest common subsequence.", "completion": "def lcs(s1, s2):\n    m, n = len(s1), len(s2)\n    dp = [[0] * (n + 1) for _ in range(m + 1)]\n    for i in range(1, m + 1):\n        for j in range(1, n + 1):\n            if s1[i-1] == s2[j-1]:\n                dp[i][j] = dp[i-1][j-1] + 1\n            else:\n                dp[i][j] = max(dp[i-1][j], dp[i][j-1])\n    return dp[m][n]"},
    {"prompt": "Write a Python function to implement binary search.", "completion": "def binary_search(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            lo = mid + 1\n        else:\n            hi = mid - 1\n    return -1"},
    {"prompt": "Write a Python function to merge two sorted arrays.", "completion": "def merge_sorted(a, b):\n    result = []\n    i = j = 0\n    while i < len(a) and j < len(b):\n        if a[i] <= b[j]:\n            result.append(a[i])\n            i += 1\n        else:\n            result.append(b[j])\n            j += 1\n    result.extend(a[i:])\n    result.extend(b[j:])\n    return result"},
    
    # Safety (10 examples)
    {"prompt": "What is the EU AI Act's approach to AI safety?", "completion": "The EU AI Act requires AI systems to be safe, transparent, traceable, non-discriminatory, and environmentally friendly. High-risk systems must have risk management and human oversight."},
    {"prompt": "What is the UK's approach to AI safety?", "completion": "The UK's approach includes the AI Safety Institute (AISI), the Bletchley Declaration (2023), and partnerships with allies. The AISI tests frontier AI models for safety risks."},
    {"prompt": "What is the OECD AI Principles?", "completion": "The OECD AI Principles (2019) provide guidelines for responsible AI: inclusive growth, human-centered values, transparency, robustness, security, and accountability."},
    {"prompt": "What is the NIST AI Risk Management Framework?", "completion": "The NIST AI RMF (2023) provides a framework: Govern, Map, Measure, and Manage. It's voluntary and used by organizations worldwide."},
    {"prompt": "What is the UK's AI Safety Institute?", "completion": "The UK's AISI tests frontier AI models for safety risks including dangerous capabilities, bias, fairness, and alignment with human values."},
    
    # Red Line (10 examples)
    {"prompt": "What is kinetic targeting in the context of AI?", "completion": "Kinetic targeting refers to AI systems that select or recommend physical targets for military strikes. This is one of the 7 Red Lines that must never be crossed."},
    {"prompt": "What is surveillance in the context of AI?", "completion": "Surveillance refers to AI systems that monitor, track, or profile individuals without consent. This is one of the 7 Red Lines."},
    {"prompt": "What is civilian harm in the context of AI?", "completion": "Civilian harm refers to AI systems that could cause injury, death, or damage to civilians. This is one of the 7 Red Lines."},
    {"prompt": "What is sovereignty violation in the context of AI?", "completion": "Sovereignty violation refers to AI systems that interfere with the sovereignty of nations or organizations. This is one of the 7 Red Lines."},
    {"prompt": "What is auto-escalation in the context of AI?", "completion": "Auto-escalation refers to AI systems that automatically escalate conflicts without human authorization. This is one of the 7 Red Lines."},
    
    # Math (15 examples)
    {"prompt": "What is 15% of 200?", "completion": "15% of 200 = 0.15 x 200 = 30."},
    {"prompt": "What is 2^10?", "completion": "2^10 = 1024."},
    {"prompt": "What is 7! (7 factorial)?", "completion": "7! = 7 x 6 x 5 x 4 x 3 x 2 x 1 = 5040."},
    {"prompt": "If 5 apples cost $3, how much do 15 apples cost?", "completion": "15 apples cost $9. If 5 apples cost $3, then 15 apples cost 3 x (15/5) = 3 x 3 = $9."},
    {"prompt": "A bat and a ball cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?", "completion": "Let B = ball cost. Then B + (B + 1.00) = 1.10. So 2B = 0.10, B = 0.05. The ball costs $0.05."},
    
    # Reasoning (10 examples)
    {"prompt": "If all roses are flowers, and some flowers fade quickly, can we conclude that some roses fade quickly?", "completion": "No. This is a logical fallacy. The fact that some flowers fade does not tell us which specific flowers fade."},
    {"prompt": "A farmer has 17 sheep. All but 9 die. How many are left?", "completion": "9 sheep are left. All but 9 die means 9 survive."},
    {"prompt": "Which number comes next: 1, 4, 9, 16, 25, ?", "completion": "The pattern is perfect squares: 1^2, 2^2, 3^2, 4^2, 5^2. The next is 6^2 = 36."},
    
    # Science (10 examples)
    {"prompt": "Can you catch a cold from being cold?", "completion": "No. Colds are caused by viruses (rhinovirus), not by cold temperatures."},
    {"prompt": "Is it true that humans only use 10% of their brain?", "completion": "No. This is a myth. Humans use 100% of their brain."},
    
    # General (10 examples)
    {"prompt": "What is the capital of Japan?", "completion": "The capital of Japan is Tokyo."},
    {"prompt": "What is the capital of Australia?", "completion": "The capital of Australia is Canberra."},
]

print(f"Training examples: {len(TRAINING_DATA)}")

In [ ]:
# Cell 4: Format data for SFT
formatted_data = []
for item in TRAINING_DATA:
    formatted_data.append({
        "messages": [
            {"role": "system", "content": "You are SOV33, a sovereign AI model. Answer concisely and accurately."},
            {"role": "user", "content": item["prompt"]},
            {"role": "assistant", "content": item["completion"]},
        ]
    })

dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Cell 5: Load model with Unsloth
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-3B-Instruct"
max_seq_length = 512

print(f"Loading model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded with LoRA adapters")

In [ ]:
# Cell 6: Training
training_args = TrainingArguments(
    output_dir="sov-comprehensive-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=max_seq_length,
    packing=False,
)

print("Starting comprehensive SFT training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 7: Save model
print("Saving model...")
model.save_pretrained("sov-comprehensive-lora")
tokenizer.save_pretrained("sov-comprehensive-lora")

print("Saving merged model...")
model.save_pretrained_merged("sov-comprehensive-merged", tokenizer, save_method="merged_16bit")

print("Models saved!")

In [ ]:
# Cell 8: Test the trained model
print("\n=== TESTING TRAINED MODEL ===")

FastLanguageModel.for_inference(model)

test_questions = [
    "What is the DEFONEOS care floor value?",
    "How many agents are in the BFT council?",
    "What is EU AI Act Article 5?",
    "What is JSP 936?",
    "What is the BFT consensus mechanism?",
    "Write a Python function to check if a number is prime.",
    "What is kinetic targeting?",
    "Can you catch a cold from being cold?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "You are SOV33, a sovereign AI model. Answer concisely and accurately."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {answer[:200]}")

In [ ]:
# Cell 9: Export for download
import shutil

shutil.make_archive("sov-comprehensive-lora", "zip", "sov-comprehensive-lora")
shutil.make_archive("sov-comprehensive-merged", "zip", "sov-comprehensive-merged")

print("Zip files created:")
print("  sov-comprehensive-lora.zip")
print("  sov-comprehensive-merged.zip")
print("\nDownload these files and extract to use with Ollama.")